# 05 — Model Comparison

Compares the environmental model (colonization pressure) and the patient model
(antibiotic exposure, age/sex-adjusted) on predictive performance, to answer the project's
core question: which better predicts hospital-onset MRSA acquisition?

**Caveat to carry into the write-up:** the two cohorts differ in matching design and
sample composition, so this is not a strictly apples-to-apples comparison — differences in
AUC partly reflect differences in the matched populations, not only differences in
predictor strength. Treat this as a scoped, directional comparison.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import RocCurveDisplay, roc_auc_score
from sklearn.model_selection import train_test_split

sys.path.append(str(Path.cwd().parent))
from src.data_loading import load_processed, drop_zero_variance

env = load_processed("environmental_mrsa.csv")
pat = load_processed("patient_mrsa.csv")

## Fit each model with an 80/20 train/test split (for fair AUC comparison)

In [ ]:
def fit_and_eval(df, predictors, label, random_state=42):
    data = df[["group_binary"] + predictors].dropna()
    X, y = data[predictors], data["group_binary"]
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=random_state
    )
    clf = LogisticRegression(max_iter=2000)
    clf.fit(X_train, y_train)
    proba = clf.predict_proba(X_test)[:, 1]
    auc = roc_auc_score(y_test, proba)
    print(f"{label}: n={len(data)}, test AUC={auc:.3f}")
    return clf, X_test, y_test, auc


cp_cols = [c for c in env.columns if c.endswith("_cp")]
env_predictors = cp_cols + ["any_surgery", "elix_index_mortality"]
env_clf, env_X_test, env_y_test, env_auc = fit_and_eval(env, env_predictors, "Environmental")

abx_cols = [c for c in pat.columns if c.endswith("_0_60") and c != "any_abx_0_60"]
abx_cols = drop_zero_variance(pat, abx_cols, outcome_col="group_binary")
pat_enc = pat.copy()
pat_enc["sex_male"] = (pat_enc["sex"].str.lower() == "male").astype(int)
pat_predictors = abx_cols + ["elix_index_mortality", "age", "sex_male"]
pat_clf, pat_X_test, pat_y_test, pat_auc = fit_and_eval(pat_enc, pat_predictors, "Patient")

## ROC curves side by side

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
RocCurveDisplay.from_estimator(env_clf, env_X_test, env_y_test, ax=ax, name="Environmental (CP)")
RocCurveDisplay.from_estimator(pat_clf, pat_X_test, pat_y_test, ax=ax, name="Patient (abx, age/sex-adj.)")
ax.plot([0, 1], [0, 1], linestyle="--", color="grey")
ax.set_title("MRSA acquisition: environmental vs. patient model")
plt.tight_layout()

## Performance summary table

In [ ]:
summary = pd.DataFrame({
    "model": ["Environmental (colonization pressure)", "Patient (antibiotic exposure, age/sex-adj.)"],
    "n": [len(env), len(pat)],
    "test_AUC": [env_auc, pat_auc],
})
summary

## Conclusion (fill in once run on real data)

State which model achieved higher AUC, then qualify with the caveat above and with the
statsmodels effect sizes/CIs from `04a`/`04b` — predictive performance and interpretability
of individual predictors are separate questions and both matter for the "which factor
matters more" framing.